# Fine-Tuning LLM Generation Models

This notebook comes from chapter 12 of the book [Hands-On Large Language Models](https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961)in which the authors detail the three main stratiges for finetuning generative LLMs using your own data under the Supervised FineTuning(SFT) approach. Namely,
*   Full FineTuning
*   Parameter-Efficient FineTuning (PEFT)
*   Preference/Alignment FineTuning through [RLHF](https://en.wikipedia.org/wiki/Reinforcement_learning_from_human_feedback#:~:text=In%20machine%20learning%2C%20reinforcement%20learning,other%20models%20through%20reinforcement%20learning.)

Of course full finetuning is like the name suggests takes a pretrained LLM and completly retrains it updating all of the LLMs weights based on your data and/or target task. While this used to be the approach for finetuning LLMs it turns out that most of the pretrained parameters do not actually need to be changed but only [*a select few need to be changed*](https://arxiv.org/pdf/1902.00751)   to achieve state of the art results for your dataset and/or task. This is in essenseu the technique of Parameter-Efficient FineTuning(PEFT).

There are two main techniques when it comes to PEFT; namely,
  1.   using Adapters
  2.   Low-Rank Adaptation (LoRA)

Adapters add *modular* and *encapsulated* layers into the transformer architecture that are retrained(also [new approaches](https://adapterhub.ml/) are adding adapters to other architectures other than transformers). Only the adapters' weights are changed while the previous pretrained weights are frozen. This allows for dynamic loading and plug-ins for various generative tasks downstream. For [transformer based architectures](https://arxiv.org/pdf/1706.03762)(i.e. BERT and all others, jk) generally the adapter layers are composed of two feedforward layers, a nonlinear transformation/function and a skip-connection. The feedforward layers project the outputs of the previous layers downwards into a lower dimensional space (i.e. latent space) in which a nonlinear function (for instance relu) is applied to the intermediate feature vector before projecting it upwards and concatinating it with the skip-connection (i.e. the orginal output the frozen transformer layer). Interestingly, where the adapters are placed within the transformer layer(s) (see houlsby vs. pfeiffer configuration) can have a *significant* impact on performance

LoRA takes a subset of a pretrained model's layers and then does full fine-tuning only on these weights and adds these weights to the frozen pretrained weights. Because generally the rank of the fine-tuned matrices are lower than the orginal pretrained matrices (hence the name low rank) it is more computationally efficient and usually LoRA is combined with quantization to even compress the LLMs even more (so they run on stuff like [this](https://github.com/AdamClarkStandke/TinyMachineLearning) lol)    

## Instruction Fine-Tuning using QLoRA

As the book details this exercise uses [TinyLlama](https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0) as the base pretrained model and I will be using the [German UltraChat](https://huggingface.co/datasets/bjoernp/ultrachat_de) version of the [Ultrachat](https://huggingface.co/datasets/HuggingFaceH4/ultrachat_200k) dataset to test out the fine-tuning performance of TinyLlama on *unseen* training data. To turn it into a german chatbot machine!!!





In [ ]:
!pip install datasets
!pip install bitsandbytes
!pip install trl

In [2]:
from transformers import pipeline
from datasets import load_dataset
import torch

In [28]:
# first step: create formatting prompt

# Load pipe to use its chat template
pipe= pipeline("text-generation", model="TinyLlama/TinyLlama-1.1B-Chat-v1.0", torch_dtype=torch.bfloat16, device_map="auto")

def format_prompt(example):
    """Format the prompt to using the <|user|> template TinyLLama is using"""
    # Format conversation
    chat = example["conversations"]
    human = chat[1:][0]['value']
    gpt = chat[1:][1]['value']
    # creating chat template/format for tinyllama
    user = "<|user|>\n"+human+"</s>\n"
    assistant = "<|assistant|>\n"+gpt+"</s>\n"
    #prompt = user + assistant
    return {"prompt": user, "completion": assistant}

# Load and format the data using the template TinyLLama is using
dataset = (
    load_dataset("bjoernp/ultrachat_de", split='train').shuffle(seed=42)
)
dataset_de = dataset.remove_columns(['len_en',"len_de", "system_prompt", "response"]).map(format_prompt)


Device set to use cuda:0


Map:   0%|          | 0/959 [00:00<?, ? examples/s]

In [31]:
print(dataset_de['prompt'][0])

<|user|>
Bitte verfasse eine umfassende und schrittweise Anleitung, die den Prozess der Fensterinstallation erklärt. Diese soll Tipps zum Messen und Ausschneiden der Öffnung, zu den Arten von Dichtungsmitteln, die benötigt werden, zu den erforderlichen Werkzeugen und zu den Sicherheitsvorkehrungen, die zu treffen sind, enthalten. Die Anleitung sollte in klarer und prägnanter Sprache verfasst sein und Diagramme oder Bilder enthalten, um das Verständnis der einzelnen Schritte zu erleichtern.</s>



In [32]:
print(dataset_de['completion'][0])

<|assistant|>
Leider kann ich als Text-basierter Assistent keine Diagramme oder Bilder bereitstellen, aber ich werde mein Bestes tun, um den Prozess der Fensterinstallation in klaren und prägnanten Schritten zu erklären.

**Schritt 1: Messen der Fensteröffnung**
Zuerst müssen Sie die Fensteröffnung messen. Verwenden Sie ein Maßband, um die Breite und Höhe der Öffnung zu messen. Messen Sie die Breite an der Oberseite, Mitte und Unterseite der Öffnung und die Höhe an beiden Seiten und in der Mitte. Verwenden Sie die kleinsten Maße, um sicherzustellen, dass das Fenster passt.

**Schritt 2: Ausschneiden der Öffnung**
Wenn Sie ein neues Fenster in eine Wand einbauen, müssen Sie die Öffnung ausschneiden. Verwenden Sie eine Stichsäge oder eine Kreissäge, um die Öffnung auszuschneiden. Stellen Sie sicher, dass die Öffnung gerade und eben ist.

**Schritt 3: Auswahl des Dichtungsmittels**
Es gibt verschiedene Arten von Dichtungsmitteln, die Sie verwenden können, darunter Silikon, Polyurethan und

In [33]:
type(dataset_de)

datasets.arrow_dataset.Dataset

In [34]:
# second step: quantization
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# 4-bit quantization configuration - Q in QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # Use 4-bit precision model loading
    bnb_4bit_quant_type="nf4",  # Quantization type
    bnb_4bit_compute_dtype="float16",  # Compute dtype
    bnb_4bit_use_double_quant=True,  # Apply nested quantization
)

# Load the model to train on the GPU
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",

    # Leave this out for regular SFT
    quantization_config=bnb_config,
)


In [35]:
# Load LLaMA tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = "<PAD>"
tokenizer.padding_side = "left"

In [36]:
# third step: LoRA config
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

# Prepare LoRA Configuration
peft_config = LoraConfig(
    lora_alpha=32,  # LoRA Scaling
    lora_dropout=0.1,  # Dropout for LoRA Layers
    r=64,  # Rank
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=  # Layers to target
     ["k_proj", "gate_proj", "v_proj", "up_proj", "q_proj", "o_proj", "down_proj"]
)

# Prepare model for training
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

In [37]:
# forth step: fine-tuning
from transformers import TrainingArguments
from trl import SFTTrainer
output_dir = "/results"

# Training arguments
training_arguments = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    num_train_epochs=1,
    logging_steps=10,
    fp16=True,
    gradient_checkpointing=True
)
# Set supervised fine-tuning parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_de,
    args=training_arguments,
    # Leave this out for regular SFT
    peft_config=peft_config,
)

# Train model
trainer.train()

# Save QLoRA weights
trainer.model.save_pretrained("TinyLlama-1.1B-qlora")

Converting train dataset to ChatML:   0%|          | 0/959 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/959 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/959 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/959 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


...

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ichglaubeya (ichglaubeya-myself) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
10,1.469400
20,1.387000
30,1.290000
40,1.270300
50,1.400800
60,1.281900
70,1.309500
80,1.319700
90,1.309000
100,1.312700


In [38]:
# fifth step: merge weights
from peft import AutoPeftModelForCausalLM

model = AutoPeftModelForCausalLM.from_pretrained(
    "TinyLlama-1.1B-qlora",
    low_cpu_mem_usage=True,
    device_map="auto",
)

# Merge LoRA and base model
merged_model = model.merge_and_unload()

In [39]:
# sixth step: profit
from transformers import pipeline

# Use our predefined prompt template
prompt = """<|user|>
Ich habe eine Frage, kannst du mir helfen</s>
<|assistant|>
"""

# Run our instruction-tuned model
pipe = pipeline(task="text-generation", model=merged_model, tokenizer=tokenizer)
print(pipe(prompt)[0]["generated_text"])

Device set to use cuda:0


<|user|>
Ich habe eine Frage, kannst du mir helfen</s>
<|assistant|>
Ich kann dir helfen, eine Frage zu stellen. Ich kann dir helfen, eine Frage zu stellen, die eine Frage darstellt, die eine Frage darstellt, die eine Frage darstellt, die eine Frage darstellt, die eine Frage darstellt, die eine Frage darstellt, die eine Frage darstellt, die eine Frage darstellt, die eine Frage darstellt, die eine Frage darstellt, die eine Frage darstellt, die eine Frage darstellt, die eine Frage darstellt, die eine Frage darstellt, die eine Frage darstellt, die eine Frage darstellt, die eine Frage darstellt, die eine Frage darstellt, die eine Frage darstellt, die eine Frage darstellt, die eine Frage darstellt, die eine Frage darstellt, die eine Frage darstellt, die eine Frage darstellt, die eine Frage darstellt, die eine Frage darstellt, die eine Frage darstellt, die eine Frage darstellt, die eine Frage darstellt, die eine Frage darstellt, die eine Frage darstellt, die eine Frage darstellt, die eine Fr